# Web Scraping coinmarketcap.com

##Installing dependencies

In [ ]:
pip install --upgrade firebase-admin

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 119 kB 4.8 MB/s 
     |████████████████████████████████| 4.1 MB 45.5 MB/s 
  Attempting uninstall: firebase-admin
    Found existing installation: firebase-admin 5.3.0
    Uninstalling firebase-admin-5.3.0:
      Successfully uninstalled firebase-admin-5.3.0


In [ ]:
pip install beautifulsoup4

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install selenium
!apt-get update # to update ubuntu to correctly run apt install
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 6.0 MB 4.8 MB/s 
     |████████████████████████████████| 384 kB 56.5 MB/s 
     |████████████████████████████████| 140 kB 57.3 MB/s 
     |████████████████████████████████| 58 kB 4.4 MB/s 
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
requests 2.23.0 requires urllib3!=1.25.0,!=1.25.1,<1.26,>=1.21.1, but you have urllib3 1.26.13 which is incompatible.


Get:1 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran40/ InRelease [3,626 B]
Ign:2 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu bionic InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  InRelease [1,581 B]
Hit:5 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  Release
Get:6 http://archive.ubuntu.com/ubuntu bionic-updates InRelease [88.7 kB]
Hit:7 http://ppa.launchpad.net/c2d4u.team/c2d4u4.0+/ubuntu bionic InRelease
Get:8 http://security.ubuntu.com/ubuntu bionic-security InRelease [88.7 kB]
Get:9 http://archive.ubuntu.com/ubuntu bionic-backports InRelease [83.3 kB]
Hit:10 http://ppa.launchpad.net/cran/libgit2/ubuntu bionic InRelease
Hit:11 http://ppa.launchpad.net/deadsnakes/ppa/ubuntu bionic InRelease
Hit:12 http://ppa.launchpad.net/graphics-drivers/ppa/ubuntu bionic InRelease
Get:13 https://developer.downlo

In [ ]:
import json
import requests
import time
from bs4 import BeautifulSoup as BS
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

##Mounting Google Drive, Authenticating firebase & Creating a firestore

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# open Firebase credentials
with open("/content/drive/MyDrive/ie_course_2022_team08/credentials/firebase_credentials.json") as f:
  credential = json.load(f)
firebase_credential = credentials.Certificate(credential)

# create Firestore database instance
firebase_admin.initialize_app(firebase_credential)
db = firestore.client()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def create_json(data):
    global file_count
    with open('crypto'+str(file_count)+'.json', 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii='False')
        file_count += 1

page = 1
file_count = 0
crypto = []
for page in range(1,28):
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    driver= webdriver.Chrome('chromedriver',options=chrome_options)
    url = "https://coinmarketcap.com/?page=" + str(page)
    driver.get(url)
    time.sleep(2)
    for i in range(1, 4):
      driver.execute_script("window.scrollTo(0, 8080)")
      time.sleep(1)
    soup= BS(driver.page_source, 'html.parser')
    blocks_tr = soup.find_all('tr')
    for item in blocks_tr:
        item_name = item.find_all('p', class_='sc-e225a64a-0 ePTNty')
        for name in item_name:
                item_MarketCap = item.find_all('span', class_='sc-b2299d0c-1 hHzHwP')
                for MarketCap in item_MarketCap:       
                    crypto.append(
                        {
                            'id' : file_count,
                            'Name' : name.text,
                            'MarketCap' : MarketCap.text
                        }
                    )
                    create_json(crypto)
                    db.collection('crypto_metadata').document('crypto-'+str(file_count)).set(crypto[0])
                    crypto = []
!cp * "drive/My Drive/ie_course_2022_team08/retrieved_data/WebScraping" 
driver.quit()


cp: -r not specified; omitting directory 'drive'
cp: -r not specified; omitting directory 'sample_data'


In [ ]:
rm *

rm: cannot remove 'drive': Is a directory
rm: cannot remove 'sample_data': Is a directory


#Text Collection From Twitter API

## Update & import Python modules

In [ ]:
!pip install tweepy

# Firebase/Firestore
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

import tweepy

# general Python modules
import csv
import re
from datetime import datetime, timedelta


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 127 kB 5.2 MB/s 
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.13
    Uninstalling urllib3-1.26.13:
      Successfully uninstalled urllib3-1.26.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
selenium 4.6.1 requires urllib3[socks]~=1.26, but you have urllib3 1.25.11 which is incompatible.


In [ ]:
with open("/content/drive/MyDrive/ie_course_2022_team08/credentials/twitter_credentials.json") as t:
  twitter_cred = json.load(t)

In [ ]:
# Consumer keys and access tokens, used for OAuth
consumer_key = twitter_cred['consumer_key']
consumer_secret = twitter_cred['consumer_secret']
access_token = twitter_cred['access_token']
access_token_secret = twitter_cred['access_token_secret']

# OAuth process, using the keys and tokens
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)

# Creation of the actual interface, using authentication
api = tweepy.API(auth, wait_on_rate_limit=True)

def create_json2(data, entity):
    with open(entity+str(month)+'-'+str(year)+'.json', 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii='False', default=str)

entities = ['MaticPolygon',
             'PolyBlocks'
           ]
month = 1
year = 2021
entity_json = {"id":[], "screen_name": [],"author_id" :[] ,"created_at":[],"text":[]}

for entity in entities:
  for year in range (2021, 2022):
    for month in range (1,13):
      for status in tweepy.Cursor(api.user_timeline, screen_name=entity, tweet_mode='extended').items():
          #print (status._json)
            if status.created_at < datetime(year, month, 28, 1, 1, 1) and status.created_at > datetime(year, month, 1, 1, 1, 1):
              entity_json["id"].append(status.id)
              entity_json["screen_name"].append(entity)
              entity_json["author_id"].append(status.id_str)
              entity_json["created_at"].append(status.created_at.strftime("%Y-%m-%d %H:%M:%S"))
              entity_json["text"].append(status.full_text)

      create_json2(entity_json, entity)
      doc= db.collection(entity).document(entity+str(month)+'-'+str(year))
      doc.set(entity_json)
      entity_json = {"id":[], "screen_name": [],"author_id" :[] ,"created_at":[],"text":[]}

!cp * "drive/My Drive/ie_course_2022_team08/retrieved_data/API"
